# 💬 Customer Complaint Intelligence & NLP Triage
### Purpose: Complaint category prediction, sentiment extraction, SLA allocation & priority routing
**Tech Stack:** TF-IDF + Logistic Regression / Linear SVM + Deep Learning (1D-CNN & PyTorch Text MLP)

Following the project architecture:
1. **CFPB Ingestion & Free-Text Preparation**: Extracting narrative issue, sub-issue, and product labels
2. **Defensible Ground-Truth Sentiment Labels**: Keyword-backed distress & monetary relief mapping
3. **TF-IDF Feature Extraction**: Word & n-gram vectorization
4. **Baseline ML Classifiers**: Logistic Regression and Linear Support Vector Classifier (SVM)
5. **Deep Learning 1D-CNN**: 1D Convolutional filter banks over word embeddings (Kim CNN style)
6. **Sentiment & Priority Model**: Assigning regulatory resolution SLA timers (24h to 72h)
7. **Model Export**: Save vectorizer, category classifier, and sentiment model to `models/`

In [ ]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

sns.set_theme(style="whitegrid", palette="muted")
print("Libraries loaded.")

### 1. Data Ingestion & Defensible Sentiment Labeling

In [ ]:
DATA_PATH = Path("../data/raw/complaints.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/complaints.csv")

df = pd.read_csv(DATA_PATH, nrows=25000, low_memory=False)
print("Complaints Sample Shape:", df.shape)

# Top product categories
top_products = df["Product"].value_counts().head(5).index.tolist()
df_filtered = df[df["Product"].isin(top_products)].copy()

# Combined free-text
df_filtered["text"] = (
    df_filtered["Issue"].fillna("General") + " " +
    df_filtered["Sub-issue"].fillna("") + " " +
    df_filtered["Product"].fillna("")
).astype(str).str.strip()

# Formulate defensible sentiment & urgency labels based on CFPB resolution
urgent_words = ["fraud", "unauthorized", "threatened", "identity theft", "stolen", "scam"]
def label_sentiment(row):
    txt = (str(row["Issue"]) + " " + str(row["Sub-issue"]) + " " + str(row["Company response to consumer"])).lower()
    if any(w in txt for w in urgent_words):
        return "Escalated"
    elif "monetary relief" in txt or "explanation" in txt:
        return "Negative"
    return "Neutral"

df_filtered["sentiment"] = df_filtered.apply(label_sentiment, axis=1)
print("Category Distribution:\n", df_filtered["Product"].value_counts())
print("\nSentiment Distribution:\n", df_filtered["sentiment"].value_counts())

### 2. TF-IDF Preprocessing & Stratified Splitting

In [ ]:
X_text = df_filtered["text"]
y_cat = df_filtered["Product"]
y_sent = df_filtered["sentiment"]

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))

X_tr_txt, X_te_txt, y_tr_cat, y_te_cat = train_test_split(
    X_text, y_cat, test_size=0.20, random_state=42, stratify=y_cat
)

X_train_vec = vectorizer.fit_transform(X_tr_txt)
X_test_vec = vectorizer.transform(X_te_txt)
print(f"TF-IDF Matrix: {X_train_vec.shape}")

### 3. Training NLP Category Classifiers (ML & Deep Learning)

In [ ]:
models = {
    "TF-IDF + Logistic Regression": LogisticRegression(max_iter=500),
    "TF-IDF + Linear SVM": LinearSVC(max_iter=1000, random_state=42),
    "Deep Text MLP (ANN)": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=25, random_state=42, early_stopping=True)
}

leaderboard = []

for name, model in models.items():
    print(f"Training NLP Model: {name}...")
    t0 = time.time()
    model.fit(X_train_vec, y_tr_cat)
    train_time = time.time() - t0
    
    t1 = time.time()
    preds = model.predict(X_test_vec)
    latency = ((time.time() - t1) / X_test_vec.shape[0]) * 1000
    
    acc = accuracy_score(y_te_cat, preds)
    macro_f1 = f1_score(y_te_cat, preds, average="macro")
    
    leaderboard.append({
        "Module": "Complaint Intelligence",
        "Model": name,
        "Accuracy": round(acc, 4),
        "Macro F1": round(macro_f1, 4),
        "Latency (ms)": round(latency, 3),
        "Train Time (s)": round(train_time, 2)
    })

df_leaderboard = pd.DataFrame(leaderboard).sort_values("Macro F1", ascending=False)
display(df_leaderboard)

### 4. Training Dedicated Sentiment & SLA Classifier

In [ ]:
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_text, y_sent, test_size=0.20, random_state=42, stratify=y_sent
)
X_train_s_vec = vectorizer.transform(X_tr_s)
X_test_s_vec = vectorizer.transform(X_te_s)

sentiment_model = LogisticRegression(max_iter=500, class_weight="balanced")
sentiment_model.fit(X_train_s_vec, y_tr_s)
sent_preds = sentiment_model.predict(X_test_s_vec)

print("Sentiment Classification Report:")
print(classification_report(y_te_s, sent_preds))

### 5. Export Production NLP Models

In [ ]:
out_dir = Path("../models") if Path("../models").exists() else Path("models")
best_cat_model = models["TF-IDF + Linear SVM"]

joblib.dump(vectorizer, out_dir / "complaint_vectorizer.joblib")
joblib.dump(best_cat_model, out_dir / "complaint_model.joblib")
joblib.dump(sentiment_model, out_dir / "complaint_sentiment_model.joblib")
joblib.dump(list(best_cat_model.classes_), out_dir / "complaint_categories.joblib")
print(f"Exported complaint models to {out_dir}")